# Validate Volume Bars Against CoinGecko

Compare our calculated WETH volume bars against CoinGecko historical volume data for selected tokens.
This notebook validates the volume bar aggregation by checking correlation with external data sources.


In [13]:
import os
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import polars as pl
import requests
from dotenv import load_dotenv

load_dotenv()

# CoinGecko API key
COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")

# Known token addresses and their CoinGecko IDs
# Note: Addresses are verified against CoinGecko API and our swap data
TOKENS_TO_VALIDATE = {
    "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48": {
        "name": "USDC",
        "coingecko_id": "usd-coin",
    },
    "0xdac17f958d2ee523a2206206994597c13d831ec7": {
        "name": "USDT",
        "coingecko_id": "tether",
    },
    "0x2260fac5e5542a773aa44fbcfedf7c193bc2c599": {
        "name": "WBTC",
        "coingecko_id": "wrapped-bitcoin",
    },
    "0x6b175474e89094c44da98b954eedeac495271d0f": {
        "name": "DAI",
        "coingecko_id": "dai",
    },
    "0x514910771af9ca656af840dff83e8264ecf986ca": {
        "name": "LINK",
        "coingecko_id": "chainlink",
    },
}

WETH_ADDRESS = "0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2"

print("Setup complete!")

Setup complete!


In [14]:
# Load volume bars and price data
df_volume_bars = pl.read_parquet("./data/weth_volume_bars.parquet")
df_prices = pl.read_parquet("./data/weth_prices_timeseries.parquet")

print(f"Loaded {len(df_volume_bars):,} volume bar records")
print(
    f"Date range: {df_volume_bars['bar_start'].min()} to {df_volume_bars['bar_end'].max()}"
)
print(f"Unique tokens: {df_volume_bars['token_address'].n_unique()}")

# Check which validation tokens exist in volume bars
validation_tokens_in_data = (
    df_volume_bars.filter(
        pl.col("token_address").is_in(list(TOKENS_TO_VALIDATE.keys()))
    )
    .select("token_address")
    .unique()
)

print(f"\nValidation tokens in volume bars: {len(validation_tokens_in_data)}")
for row in validation_tokens_in_data.to_dicts():
    token_addr = row["token_address"]
    token_info = TOKENS_TO_VALIDATE.get(token_addr)
    if token_info:
        print(f"  ✓ {token_info['name']}")
    else:
        print(f"  ? {token_addr[:8]}...")

Loaded 991,326 volume bar records
Date range: 2025-07-01 00:00:11 to 2025-09-01 23:59:59
Unique tokens: 660

Validation tokens in volume bars: 5
  ✓ USDT
  ✓ LINK
  ✓ WBTC
  ✓ USDC
  ✓ DAI


In [15]:
def fetch_coingecko_volume_and_price(
    token_id: str, start_date: str, end_date: str
) -> dict:
    """Fetch historical volume and price from CoinGecko API.

    Args:
        token_id: CoinGecko token ID (e.g., 'usd-coin')
        start_date: Start date in YYYY-MM-DD format
        end_date: End date in YYYY-MM-DD format

    Returns:
        Dictionary with timestamps, volumes (USD), and prices (ETH)
    """
    # Convert dates to timestamps
    start_ts = int(
        datetime.strptime(start_date, "%Y-%m-%d")
        .replace(tzinfo=timezone.utc)
        .timestamp()
    )
    end_ts = int(
        datetime.strptime(end_date, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp()
    )

    # Use market_chart/range endpoint to get volume and price data
    url = f"https://api.coingecko.com/api/v3/coins/{token_id}/market_chart/range"
    params = {
        "vs_currency": "eth",
        "from": start_ts,
        "to": end_ts,
    }
    headers = {
        "x-cg-demo-api-key": COINGECKO_API_KEY,
    }

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    data = response.json()
    prices = data.get("prices", [])
    volumes = data.get("total_volumes", [])

    # Convert to polars DataFrame
    # Note: total_volumes from CoinGecko is in the target currency (ETH in this case)
    df = pl.DataFrame(
        {
            "timestamp": [
                datetime.fromtimestamp(p[0] / 1000, tz=timezone.utc) for p in prices
            ],
            "price_eth": [p[1] for p in prices],
            "volume_eth": [v[1] for v in volumes],  # Already in ETH
        }
    )

    return df


print("Helper function defined!")

Helper function defined!


In [16]:
# Fetch CoinGecko data for validation tokens
coingecko_data = {}

for address, info in TOKENS_TO_VALIDATE.items():
    print(f"Fetching {info['name']} volume and price from CoinGecko...")
    try:
        df_cg = fetch_coingecko_volume_and_price(
            info["coingecko_id"], "2025-07-01", "2025-09-01"
        )
        coingecko_data[address] = df_cg
        print(
            f"  Got {len(df_cg)} data points (vol: {df_cg['volume_eth'].sum():.2f} ETH)"
        )
    except Exception as e:
        print(f"  Error: {e}")

print("\nDone fetching CoinGecko data!")

Fetching USDC volume and price from CoinGecko...
  Got 1488 data points (vol: 4607580961.66 ETH)
Fetching USDT volume and price from CoinGecko...
  Got 1488 data points (vol: 4607580961.66 ETH)
Fetching USDT volume and price from CoinGecko...
  Got 1489 data points (vol: 38290402748.31 ETH)
Fetching WBTC volume and price from CoinGecko...
  Got 1489 data points (vol: 38290402748.31 ETH)
Fetching WBTC volume and price from CoinGecko...
  Got 1488 data points (vol: 91102502.79 ETH)
Fetching DAI volume and price from CoinGecko...
  Got 1488 data points (vol: 91102502.79 ETH)
Fetching DAI volume and price from CoinGecko...
  Got 1489 data points (vol: 45417370.75 ETH)
Fetching LINK volume and price from CoinGecko...
  Got 1489 data points (vol: 45417370.75 ETH)
Fetching LINK volume and price from CoinGecko...
  Got 1489 data points (vol: 418085573.05 ETH)

Done fetching CoinGecko data!
  Got 1489 data points (vol: 418085573.05 ETH)

Done fetching CoinGecko data!


In [19]:
# Compare volumes for each token
print("=" * 80)
print("VOLUME VALIDATION SUMMARY")
print("=" * 80)
print("\nNote: CoinGecko volumes are market-wide aggregates across all exchanges.")
print(
    "Our volume bars capture only WETH-paired swaps from Uniswap V3 (filtered subset)."
)
print(
    "We validate that daily volumes *correlate* (both track market trends),\nnot that they match in magnitude.\n"
)

validation_results = []

for address, info in TOKENS_TO_VALIDATE.items():
    token_name = info["name"]
    print(f"{token_name}:")

    # Get our volume bars for this token
    df_our_bars = df_volume_bars.filter(pl.col("token_address") == address)

    if len(df_our_bars) == 0:
        print("  ⚠ No volume bars in our data\n")
        continue

    if address not in coingecko_data:
        print("  ⚠ No CoinGecko data fetched\n")
        continue

    df_cg = coingecko_data[address]

    # Aggregate our bars to calendar-day volumes
    df_our_daily = (
        df_our_bars.with_columns(pl.col("bar_start").dt.truncate("1d").alias("date"))
        .group_by("date")
        .agg(
            [
                pl.col("realized_volume").sum().alias("our_volume"),
                pl.col("close_price").mean().alias("avg_price"),
            ]
        )
        .with_columns(pl.col("date").cast(pl.Date))
        .sort("date")
    )

    # Aggregate CoinGecko to calendar-day (daily 24h volume)
    df_cg_daily = (
        df_cg.with_columns(pl.col("timestamp").dt.truncate("1d").alias("date"))
        .group_by("date")
        .agg(
            [
                pl.col("volume_eth").mean().alias("cg_volume"),
                pl.col("price_eth").mean().alias("cg_price"),
            ]
        )
        .with_columns(pl.col("date").cast(pl.Date))
        .sort("date")
    )

    # Join on calendar date
    df_joined = df_our_daily.join(df_cg_daily, on="date", how="inner")

    if len(df_joined) == 0:
        print("  ⚠ No overlapping dates\n")
        continue

    # Core validation metric: correlation
    corr = df_joined.select(pl.corr("our_volume", "cg_volume").alias("corr"))["corr"][0]

    # Price correlation (should be very high since price is a point estimate)
    price_corr = df_joined.select(pl.corr("avg_price", "cg_price").alias("corr"))[
        "corr"
    ][0]
    price_mape = (
        ((df_joined["avg_price"] - df_joined["cg_price"]).abs())
        / df_joined["cg_price"]
        * 100
    ).mean()

    # Magnitude comparison (for reference)
    our_total = df_joined["our_volume"].sum()
    cg_total = df_joined["cg_volume"].sum()
    volume_ratio = our_total / cg_total if cg_total > 0 else float("nan")

    print(f"  Period: {df_joined['date'].min()} to {df_joined['date'].max()}")
    print(f"  Days matched: {len(df_joined)}")
    print(f"  Volume correlation: {corr:.4f}")
    print(f"  Price correlation: {price_corr:.4f}")
    print(f"  Price MAPE: {price_mape:.2f}%")
    print(
        f"  Volume totals: {our_total:.0f} WETH (ours) vs {cg_total:.0f} ETH (CoinGecko)"
    )
    print(f"  Volume ratio (ours/CG): {volume_ratio:.4f}")

    # Show a few sample days
    samples = df_joined.select(["date", "our_volume", "cg_volume"]).head(3).to_dicts()
    print(f"  Sample days:")
    for row in samples:
        print(
            f"    {row['date']}: ours={row['our_volume']:.2f}, CG={row['cg_volume']:.2f}"
        )

    validation_results.append(
        {
            "token": token_name,
            "volume_correlation": float(corr),
            "price_correlation": float(price_corr),
            "price_mape": float(price_mape),
        }
    )
    print()

print("=" * 80)
print("\nInterpretation:")
print("  - Volume correlation > 0.5: Your bars track market trends well ✓")
print("  - Price correlation > 0.95: Your prices align with external data ✓")
print("  - Price MAPE < 5%: Prices are accurate ✓")

VOLUME VALIDATION SUMMARY

Note: CoinGecko volumes are market-wide aggregates across all exchanges.
Our volume bars capture only WETH-paired swaps from Uniswap V3 (filtered subset).
We validate that daily volumes *correlate* (both track market trends),
not that they match in magnitude.

USDC:
  Period: 2025-07-01 to 2025-08-31
  Days matched: 62
  Volume correlation: 0.5305
  Price correlation: 0.9984
  Price MAPE: 0.92%
  Volume totals: 3791456 WETH (ours) vs 191899366 ETH (CoinGecko)
  Volume ratio (ours/CG): 0.0198
  Sample days:
    2025-07-01: ours=43281.46, CG=2709483.68
    2025-07-02: ours=51937.75, CG=2369198.99
    2025-07-03: ours=51937.75, CG=3258970.60

USDT:
  Period: 2025-07-01 to 2025-08-31
  Days matched: 62
  Volume correlation: 0.3883
  Price correlation: 0.9984
  Price MAPE: 0.95%
  Volume totals: 4246473 WETH (ours) vs 1594062281 ETH (CoinGecko)
  Volume ratio (ours/CG): 0.0027
  Sample days:
    2025-07-01: ours=48922.49, CG=13381228.81
    2025-07-02: ours=48922.

In [ ]:
    # Convert to daily
    df_our_daily = (
        df_our_bars.with_columns(pl.col("bar_start").dt.truncate("1d").alias("date"))
        .group_by("date")
        .agg(
            [
                pl.col("realized_volume").sum().alias("total_volume"),
                pl.col("close_price").mean().alias("avg_price"),
            ]
        )
        .with_columns(pl.col("date").cast(pl.Date))
        .sort("date")
    )

    df_cg_daily = (
        df_cg.with_columns(pl.col("timestamp").dt.truncate("1d").alias("date"))
        .group_by("date")
        .agg(
            [
                pl.col("volume_eth").mean().alias("cg_volume"),
                pl.col("price_eth").mean().alias("cg_price"),
            ]
        )
        .with_columns(pl.col("date").cast(pl.Date))
        .sort("date")
    )

    df_joined = df_our_daily.join(df_cg_daily, on="date", how="inner")

In [20]:
# Summary statistics and sanity checks
print("\n" + "=" * 80)
print("DATA QUALITY CHECKS")
print("=" * 80)

# Check for nulls
print("\nNull value checks:")
null_counts = df_volume_bars.null_count()
for col in null_counts.columns:
    count = null_counts[col][0]
    if count > 0:
        print(f"  ⚠ {col}: {count} nulls")
    else:
        print(f"  ✓ {col}: no nulls")

# Check for negative volumes
negative_vols = df_volume_bars.filter(pl.col("realized_volume") < 0)
if len(negative_vols) > 0:
    print(f"\n⚠ Found {len(negative_vols)} bars with negative volume!")
else:
    print("\n✓ All volumes are non-negative")

# Check for zero volumes
zero_vols = df_volume_bars.filter(pl.col("realized_volume") == 0)
print(
    f"  Zero-volume bars: {len(zero_vols)} ({len(zero_vols) / len(df_volume_bars) * 100:.2f}%)"
)

# Check bar_end >= bar_start
time_order = df_volume_bars.filter(pl.col("bar_end") < pl.col("bar_start"))
if len(time_order) > 0:
    print(f"\n⚠ Found {len(time_order)} bars where bar_end < bar_start!")
else:
    print("\n✓ All bars have bar_end >= bar_start")

# Check for outliers in prices
print("\n" + "=" * 80)
print("PRICE STATISTICS")
print("=" * 80)

for col in ["open_price", "high_price", "low_price", "close_price"]:
    stats = df_volume_bars.select(
        [
            pl.col(col).min().alias("min"),
            pl.col(col).max().alias("max"),
            pl.col(col).mean().alias("mean"),
            pl.col(col).std().alias("std"),
        ]
    )
    print(f"\n{col}:")
    for stat_col in stats.columns:
        val = stats[stat_col][0]
        print(f"  {stat_col}: {val:.6f}")

print("\n" + "=" * 80)
print("VOLUME STATISTICS")
print("=" * 80)

vol_stats = df_volume_bars.select(
    [
        pl.col("realized_volume").min().alias("min"),
        pl.col("realized_volume").max().alias("max"),
        pl.col("realized_volume").mean().alias("mean"),
        pl.col("realized_volume").median().alias("median"),
        pl.col("realized_volume").std().alias("std"),
        (pl.col("realized_volume") > 0).sum().alias("non_zero_count"),
    ]
)

for col in vol_stats.columns:
    val = vol_stats[col][0]
    print(f"{col}: {val:,.6f}")

print(f"\nTotal records: {len(df_volume_bars):,}")


DATA QUALITY CHECKS

Null value checks:
  ✓ token_address: no nulls
  ✓ bar_start: no nulls
  ✓ bar_end: no nulls
  ✓ open_price: no nulls
  ✓ high_price: no nulls
  ✓ low_price: no nulls
  ✓ close_price: no nulls
  ✓ realized_volume: no nulls

✓ All volumes are non-negative
  Zero-volume bars: 0 (0.00%)

✓ All bars have bar_end >= bar_start

PRICE STATISTICS

open_price:
  min: 0.000000
  max: 2163173.086236
  mean: 2.435103
  std: 2172.763190

high_price:
  min: 0.000000
  max: 3567792.670615
  mean: 87.280607
  std: 15573.664461

low_price:
  min: 0.000000
  max: 75.735008
  mean: 0.222604
  std: 2.465508

close_price:
  min: 0.000000
  max: 2163173.086236
  mean: 2.434992
  std: 2172.763188

VOLUME STATISTICS
min: 0.000000
max: 9,784.498890
mean: 10.884492
median: 0.004903
std: 277.215623
non_zero_count: 991,326.000000

Total records: 991,326
